In [3]:
"""
Improved MCI-to-AD Conversion Prediction
Generates latent features for JMBayes2 joint modeling in R
Key Enhancements:
1. Efron's Cox loss for tied event times
2. Future MMSE prediction (not current)
3. Enhanced clinical features
4. No data leakage: scaler fitted on train split only
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import numpy as np
import os
import pickle
from lifelines.utils import concordance_index
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================
CONFIG = {
    'latent_dim': 128,
    'img_out_dim': 256,
    'tab_out_dim': 64,
    'lstm_hidden': 128,
    'lstm_layers': 2,
    'dropout': 0.3,
    'lr': 5e-4,
    'weight_decay': 1e-4,
    'epochs': 100,
    'batch_size': 16,
    'accumulation_steps': 1,
    'max_seq_len': 10,
    'alpha_survival': 0.85,
    'alpha_mmse': 0.15,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

# ============================================================================
# FEATURE DEFINITIONS
# ============================================================================
DEMOGRAPHIC_COLUMNS = [
    'AGE', 'PTGENDER_encoded', 'PTEDUCAT', 'PTETHCAT_encoded',
    'PTRACCAT_encoded', 'PTMARRY_encoded'
]
STATIC_FEATURES = [
    'age_bl', 'PTGENDER_encoded', 'PTEDUCAT', 'PTETHCAT_encoded',
    'PTRACCAT_encoded', 'PTMARRY_encoded'
]
TEMPORAL_FEATURES = [
    'time_from_baseline', 'AGE', 'age_since_bl',
    'mmse_slope', 'visit_number', 'MMSE', 'ADAS13',
    'mmse_variability', 'adas_mmse_discordance', 'mmse_acceleration'
]

# ============================================================================
# FEATURE ENGINEERING
# ============================================================================
def add_future_mmse_label(df, horizon=1.0, window=0.25):
    """Add MMSE score ~12 months later (no leakage: only looks forward in time)"""
    # ✅ SAFETY CHECK: ensure df contains only ONE patient
    if "PTID" in df.columns:
        assert df["PTID"].nunique() == 1, "Data leakage risk: df contains multiple patients!"
        
    future_mmse = []
    for i, row in df.iterrows():
        current_time = row["Years_bl"]
        target_time = current_time + horizon
        mask = (
            (df["Years_bl"] >= target_time - window) &
            (df["Years_bl"] <= target_time + window)
        )
        candidates = df[mask]
        if len(candidates) > 0:
            future_mmse.append(candidates.iloc[(candidates["Years_bl"] - target_time).abs().argmin()]["MMSE"])
        else:
            future_mmse.append(np.nan)
    df["MMSE_future12"] = future_mmse
    return df


def engineer_features(df):
    """Engineer temporal features using only past data at each time step."""
    df = df.sort_values("Years_bl").reset_index(drop=True)

    df["time_from_baseline"] = df["Years_bl"] - df["Years_bl"].iloc[0]
    df["age_bl"]             = df["AGE"].iloc[0]
    df["age_since_bl"]       = df["AGE"] - df["age_bl"]
    df["visit_number"]       = range(len(df))

    df["mmse_slope"] = 0.0
    df["mmse_acceleration"] = 0.0
    df["mmse_variability"] = 0.0
    df["adas_mmse_discordance"] = 0.0

    for t in range(len(df)):
        df_past = df.iloc[:t + 1]

        # MMSE slope (t vs t-1 only)
        if len(df_past) >= 2:
            dy = df_past["MMSE"].iloc[-1] - df_past["MMSE"].iloc[-2]
            dt = df_past["Years_bl"].iloc[-1] - df_past["Years_bl"].iloc[-2]
            df.loc[t, "mmse_slope"] = dy / dt if dt != 0 else 0.0

        # MMSE acceleration (change in slope)
        if t >= 2:
            df.loc[t, "mmse_acceleration"] = df.loc[t, "mmse_slope"] - df.loc[t - 1, "mmse_slope"]

        # MMSE variability (std of past visits only)
        if len(df_past) >= 2:
            df.loc[t, "mmse_variability"] = df_past["MMSE"].std()

        # ADAS–MMSE discordance
        df.loc[t, "adas_mmse_discordance"] = abs(
            df_past["ADAS13"].iloc[-1] - df_past["MMSE"].iloc[-1]
        )

    df = df.ffill()
    df = df.fillna(0)
    df = add_future_mmse_label(df)
    return df


# ============================================================================
# DATASET  (raw — NO scaling here)
# ============================================================================
class SequenceDataset(Dataset):
    """
    Stores raw (unscaled) tabular features.
    Scaling is applied externally in main() after the train/val split
    so there is zero leakage from val into the scaler statistics.
    """
    def __init__(self, manifest, valid_patients, transform=None, max_seq_len=10):
        self.transform = transform
        self.max_seq_len = max_seq_len
        self.sequences = []
        self.scaler = None   # set externally after split

        manifest = manifest.copy()
        manifest["path"] = manifest["path"].str.replace("\\", "/", regex=False)
        manifest["path"] = "./AD_Multimodal/TFN_AD/" + manifest["path"]

        skipped_not_valid = 0
        skipped_errors = 0
        processed = 0

        for ptid in manifest["PTID"].unique():
            if ptid not in valid_patients:
                skipped_not_valid += 1
                continue

            # ----------------------------------------------------------------
            # Surface exceptions instead of silently swallowing them.
            # Switch the try/except back once you're confident the data is clean.
            # ----------------------------------------------------------------
            try:
                patient_rows = manifest[manifest["PTID"] == ptid]
                if len(patient_rows) == 0:
                    continue

                df = pd.read_pickle(patient_rows.iloc[0]["path"])
                df = engineer_features(df)

                dx_seq = df["DX"].tolist()
                if "MCI" not in dx_seq:
                    continue

                mci_idx = dx_seq.index("MCI")
                ad_idx = next(
                    (i for i, x in enumerate(dx_seq[mci_idx + 1:], start=mci_idx + 1)
                     if x in ["AD", "Dementia"]),
                    -1
                )

                if ad_idx != -1:
                    time_to_event = (
                        df["Years_bl"].iloc[ad_idx] - df["Years_bl"].iloc[mci_idx]
                    )
                    event = 1
                else:
                    time_to_event = (
                        df["Years_bl"].iloc[-1] - df["Years_bl"].iloc[mci_idx]
                    )
                    event = 0

                imgs, tabs, times, mmse_vals, mmse_future = [], [], [], [], []
                valid_visits = 0

                for _, visit in df.iterrows():
                    image_path = visit["image_path"].replace(
                        "/home/mason/ADNI_Dataset/",
                        "./AD_Multimodal/ADNI_Dataset/"
                    )
                    if not os.path.exists(image_path):
                        continue

                    img = Image.open(image_path).convert("RGB")
                    if self.transform:
                        img = self.transform(img)
                    imgs.append(img)

                    # Collect only the features that actually exist in the df.
                    # Missing columns are filled with 0 to avoid silent KeyErrors.
                    feature_cols = TEMPORAL_FEATURES + STATIC_FEATURES
                    row_vals = []
                    for col in feature_cols:
                        row_vals.append(float(visit[col]) if col in visit.index else 0.0)
                    tabs.append(np.array(row_vals, dtype=np.float32))

                    times.append(float(visit["Years_bl"]))
                    mmse_vals.append(float(visit["MMSE"]))
                    future_val = visit["MMSE_future12"] if "MMSE_future12" in visit.index else np.nan
                    mmse_future.append(float(future_val) if pd.notna(future_val) else np.nan)

                    valid_visits += 1
                    if valid_visits >= self.max_seq_len:
                        break

                if valid_visits < 2:
                    continue

                # Pad sequences
                pad_len = self.max_seq_len - valid_visits
                for _ in range(pad_len):
                    imgs.append(torch.zeros_like(imgs[-1]))
                    tabs.append(np.zeros_like(tabs[-1]))
                    times.append(times[-1])
                    mmse_vals.append(np.nan)
                    mmse_future.append(np.nan)

                self.sequences.append({
                    "ptid": ptid,
                    "imgs": torch.stack(imgs),
                    "tabs": np.array(tabs, dtype=np.float32),   # RAW, unscaled
                    "times": np.array(times, dtype=np.float32),
                    "mmse": np.array(mmse_vals, dtype=np.float32),
                    "mmse_future": np.array(mmse_future, dtype=np.float32),
                    "seq_len": valid_visits,
                    "time_to_event": float(time_to_event),
                    "event": int(event),
                })
                processed += 1

            except Exception as e:
                skipped_errors += 1
                print(f"  ⚠️  Skipped patient {ptid}: {type(e).__name__}: {e}")
                continue

        print(f"  Processed : {processed} valid patients")
        print(f"  Skipped (not in valid set) : {skipped_not_valid}")
        print(f"  Skipped (errors)           : {skipped_errors}")

    def apply_scaler(self, scaler):
        self.scaler = scaler
        for seq in self.sequences:
            real_len = seq['seq_len']
            
            # ONLY scale real visits
            seq['tabs'][:real_len] = scaler.transform(seq['tabs'][:real_len])
            
            # Keep padding as zeros
            if real_len < len(seq['tabs']):
                seq['tabs'][real_len:] = 0.0

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        return (
            seq['imgs'],
            seq['tabs'],
            seq['times'],
            seq['mmse'],
            seq['mmse_future'],
            seq['seq_len'],
            seq['time_to_event'],
            seq['event'],
            seq['ptid'],
        )


# ============================================================================
# MODEL
# ============================================================================
class TensorFusion(nn.Module):
    def __init__(self, v_dim, d_dim, t_dim, proj_dim=None, dropout=0.1):
        super().__init__()
        self.output_dim = (v_dim + 1) * (d_dim + 1) * (t_dim + 1)
        self.dropout = nn.Dropout(dropout)
        self.proj = nn.Linear(self.output_dim, proj_dim) if proj_dim else None

    def forward(self, v, d, t):
        bs = v.shape[0]
        v1 = torch.cat([v, torch.ones(bs, 1, device=v.device)], dim=1)
        d1 = torch.cat([d, torch.ones(bs, 1, device=d.device)], dim=1)
        t1 = torch.cat([t, torch.ones(bs, 1, device=t.device)], dim=1)
        fusion = torch.einsum('bi,bj,bk->bijk', v1, d1, t1).view(bs, -1)
        fusion = self.dropout(fusion)
        return self.proj(fusion) if self.proj else fusion


class AttentionImageEncoder(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        base = models.resnet18(pretrained=True)
        for param in list(base.parameters())[:-20]:
            param.requires_grad = False
        self.features = nn.Sequential(*list(base.children())[:-2])
        self.attention = nn.Sequential(
            nn.Conv2d(512, 256, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 1, 1), nn.Sigmoid()
        )
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Linear(512, out_dim)

    def forward(self, x):
        feats = self.features(x)
        feats = feats * self.attention(feats)
        return self.proj(self.global_pool(feats).view(x.size(0), -1))


class SupervisedTemporalFusionAutoencoder(nn.Module):
    def __init__(self, tab_dim, config):
        super().__init__()
        self.config = config

        self.img_encoder = AttentionImageEncoder(out_dim=config['img_out_dim'])
        self.tab_encoder = nn.Sequential(
            nn.Linear(tab_dim, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Dropout(config['dropout']),
            nn.Linear(128, config['tab_out_dim']), nn.ReLU()
        )
        self.time_encoder = nn.Sequential(nn.Linear(1, 16), nn.ReLU())

        self.fusion = TensorFusion(
            v_dim=config['img_out_dim'],
            d_dim=config['tab_out_dim'],
            t_dim=16,
            dropout=config['dropout']
        )

        fusion_dim = (config['img_out_dim'] + 1) * (config['tab_out_dim'] + 1) * 17
        self.fusion_proj = nn.Sequential(
            nn.Linear(fusion_dim, config['latent_dim']),
            nn.BatchNorm1d(config['latent_dim']), nn.ReLU()
        )

        self.lstm = nn.LSTM(
            input_size=config['latent_dim'],
            hidden_size=config['lstm_hidden'],
            num_layers=config['lstm_layers'],
            batch_first=True,
            dropout=config['dropout'] if config['lstm_layers'] > 1 else 0,
            bidirectional=True
        )
        self.temporal_proj = nn.Sequential(
            nn.Linear(config['lstm_hidden'] * 2, config['latent_dim']), nn.ReLU()
        )
        self.survival_head = nn.Sequential(
            nn.Linear(config['latent_dim'], 64), nn.ReLU(),
            nn.Dropout(config['dropout']),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )
        self.mmse_head = nn.Sequential(
            nn.Linear(config['latent_dim'], 32), nn.ReLU(), nn.Linear(32, 1)
        )

    def encode_visit(self, img, tab, time):
        v = self.img_encoder(img)
        d = self.tab_encoder(tab)
        t = self.time_encoder(time.unsqueeze(1))
        z = self.fusion(v, d, t)
        return self.fusion_proj(z.view(z.size(0), -1))

    def forward(self, img_seq, tab_seq, time_seq, seq_lengths):
        bs, seq_len = img_seq.shape[:2]
        z_list = [
            self.encode_visit(img_seq[:, t], tab_seq[:, t], time_seq[:, t])
            for t in range(seq_len)
        ]
        z_seq = torch.stack(z_list, dim=1)

        packed = nn.utils.rnn.pack_padded_sequence(
            z_seq, seq_lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        lstm_out, (h_n, _) = self.lstm(packed)

        h_final = torch.cat([h_n[-2], h_n[-1]], dim=1)
        z_final = self.temporal_proj(h_final)

        return z_final, self.survival_head(z_final), self.mmse_head(z_final)


# ============================================================================
# LOSS
# ============================================================================
def cox_partial_likelihood_loss_efron(risk_scores, times, events):
    device = risk_scores.device
    mask = torch.isfinite(risk_scores) & torch.isfinite(times)
    risk_scores, times, events = risk_scores[mask], times[mask], events[mask]

    if events.sum() == 0:
        return torch.tensor(0.0, device=device)

    order = torch.argsort(times)
    risk_scores, times, events = risk_scores[order], times[order], events[order]

    log_risk = risk_scores.view(-1)
    hazard = torch.exp(log_risk)

    loss = torch.tensor(0.0, device=device)
    for t in torch.unique(times[events == 1]):
        at_risk = times >= t
        died = (times == t) & (events == 1)
        if died.sum() == 0:
            continue
        n_died = died.sum().float()
        risk_set = hazard[at_risk].sum()
        tied_risk = hazard[died].sum()
        for j in range(int(n_died)):
            loss += torch.log(risk_set - (j / n_died) * tied_risk + 1e-7)
        loss -= log_risk[died].sum()

    return loss / (events.sum() + 1e-7)


# ============================================================================
# TRAINING & VALIDATION
# ============================================================================
def train_epoch(model, loader, optimizer, config, device):
    model.train()
    total_loss = total_cox = total_mmse = 0.0

    for batch in loader:
        imgs, tabs, times, mmse, mmse_future, seq_lens, t_event, event, _ = batch

        imgs = imgs.to(device)
        tabs = torch.FloatTensor(tabs).to(device)
        times = torch.FloatTensor(times).to(device)
        mmse_future_vals = torch.FloatTensor(mmse_future).to(device)
        seq_lens = torch.LongTensor(seq_lens)
        t_event = t_event.float().to(device)
        event = event.float().to(device)

        # Use the FUTURE MMSE at the last real visit as target
        mmse_targets = torch.stack([
            mmse_future_vals[i, seq_lens[i] - 1]
            for i in range(len(seq_lens))
        ]).to(device)

        z_final, risk_scores, mmse_pred = model(imgs, tabs, times, seq_lens)

        loss_cox = cox_partial_likelihood_loss_efron(
            risk_scores.squeeze(), t_event, event
        )

        valid_mask = ~torch.isnan(mmse_targets)
        loss_mmse = (
            F.mse_loss(mmse_pred.squeeze()[valid_mask], mmse_targets[valid_mask])
            if valid_mask.any() else torch.tensor(0.0, device=device)
        )

        loss = config['alpha_survival'] * loss_cox + config['alpha_mmse'] * loss_mmse

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        total_cox += loss_cox.item()
        total_mmse += loss_mmse.item() if valid_mask.any() else 0.0

    n = len(loader)
    return {'total': total_loss / n, 'cox': total_cox / n, 'mmse': total_mmse / n}


def validate(model, loader, device):
    model.eval()
    all_risks, all_times, all_events = [], [], []

    with torch.no_grad():
        for batch in loader:
            imgs, tabs, times, _, _, seq_lens, t_event, event, _ = batch
            imgs = imgs.to(device)
            tabs = torch.FloatTensor(tabs).to(device)
            times = torch.FloatTensor(times).to(device)
            seq_lens = torch.LongTensor(seq_lens)

            _, risk_scores, _ = model(imgs, tabs, times, seq_lens)
            all_risks.extend(risk_scores.cpu().numpy().flatten())
            all_times.extend(t_event.numpy())
            all_events.extend(event.numpy())

    all_events = np.array(all_events).astype(bool)
    c_index = concordance_index(np.array(all_times), -np.array(all_risks), all_events)
    return c_index, np.array(all_risks), np.array(all_times), all_events


# ============================================================================
# EXPORT
# ============================================================================
def export_latent_features(model, loader, device, output_path):
    model.eval()
    rows = []
    BASELINE_FEATURES = ['AGE', 'PTGENDER', 'PTEDUCAT', 'ADAS13']
    tab_feature_names = TEMPORAL_FEATURES + STATIC_FEATURES

    with torch.no_grad():
        for batch in loader:
            imgs, tabs, times, mmse, mmse_future, seq_lens, t_event, event, ptids = batch
            imgs = imgs.to(device)
            tabs = tabs.to(device)
            times = times.to(device)
            seq_lens = seq_lens.cpu().long()

            for i in range(len(ptids)):
                slen = seq_lens[i].item()
                for t in range(slen):
                    img_t = imgs[i:i + 1, t]
                    tab_t = tabs[i:i + 1, t]
                    time_t = times[i:i + 1, t]

                    z_visit = model.encode_visit(img_t, tab_t, time_t)

                    # Inverse-transform to recover original scale for metadata cols.
                    # loader.dataset is always the base SequenceDataset here
                    # (full_loader wraps it directly, not via a Subset).
                    base_dataset = (
                        loader.dataset.dataset
                        if hasattr(loader.dataset, 'dataset')
                        else loader.dataset
                    )
                    tab_vals_unscaled = base_dataset.scaler.inverse_transform(
                        tabs[i, t].cpu().numpy().reshape(1, -1)
                    )[0]

                    row = {
                        "PTID": ptids[i],
                        "Years_bl": float(times[i, t].cpu()),
                        "MMSE": float(mmse[i, t]),
                        "time_to_event": float(t_event[i]),
                        "event": int(event[i]),
                    }

                    for feat in BASELINE_FEATURES:
                        feat_encoded = feat + '_encoded' if feat == 'PTGENDER' else feat
                        if feat_encoded in tab_feature_names:
                            idx = tab_feature_names.index(feat_encoded)
                            row[feat] = float(tab_vals_unscaled[idx])

                    for k, val in enumerate(z_visit[0].cpu().numpy()):
                        row[f"z_{k}"] = float(val)

                    rows.append(row)

    df = pd.DataFrame(rows).sort_values(["PTID", "Years_bl"])
    df.to_csv(output_path, index=False)
    print(f"\n✓ Exported to {output_path}")
    print(f"  Patients        : {df['PTID'].nunique()}")
    print(f"  Visits          : {len(df)}")
    print(f"  Latent features : {len([c for c in df.columns if c.startswith('z_')])}")
    return df


def export_patient_level_latent(model, loader, device, output_path):
    """
    Export one row per patient containing:
      - z_final_0 ... z_final_N  : LSTM final hidden state (the real patient summary)
      - z_slope_0 ... z_slope_N  : slope of each latent dim across visits (trajectory signal)
      - z_last_0  ... z_last_N   : encoding at the LAST real visit
      - risk_score               : neural net's own survival risk score
      - MMSE, Years_bl, time_to_event, event from last real visit
    
    This preserves EVERYTHING the LSTM learned about the patient trajectory
    instead of collapsing to visit-1 features.
    """
    model.eval()
    rows = []
 
    with torch.no_grad():
        for batch in loader:
            imgs, tabs, times, mmse, mmse_future, seq_lens, t_event, event, ptids = batch
 
            imgs      = imgs.to(device)
            tabs      = tabs.to(device)
            times_dev = times.to(device)
            seq_lens_t = torch.LongTensor(seq_lens)
 
            # z_final = LSTM's integrated patient summary (shape: bs x latent_dim)
            # This is what the survival head was trained on
            z_final, risk_scores, _ = model(imgs, tabs, times_dev, seq_lens_t)
 
            # Also get per-visit encodings to compute trajectory slopes
            bs, seq_len = imgs.shape[:2]
            z_visits = []
            for t in range(seq_len):
                z_t = model.encode_visit(
                    imgs[:, t],
                    tabs[:, t],
                    times_dev[:, t]
                )
                z_visits.append(z_t.cpu().numpy())  # (bs, latent_dim)
 
            z_visits = np.array(z_visits)  # (seq_len, bs, latent_dim)
 
            for i in range(len(ptids)):
                slen = int(seq_lens[i])
                row = {
                    "PTID":          ptids[i],
                    "time_to_event": float(t_event[i]),
                    "event":         int(event[i]),
                    "risk_score":    float(risk_scores[i].cpu()),
                    # MMSE at last real visit (for joint model longitudinal submodel)
                    "MMSE":          float(mmse[i, slen - 1]),
                    "Years_bl":      float(times[i, slen - 1]),
                    "n_visits":      slen,
                }
 
                # 1. z_final: LSTM's integrated summary — primary Cox features
                for k, val in enumerate(z_final[i].cpu().numpy()):
                    row[f"z_final_{k}"] = float(val)
 
                # 2. z_last: encoding at final visit — captures most recent state
                z_last_visit = z_visits[slen - 1, i, :]
                for k, val in enumerate(z_last_visit):
                    row[f"z_last_{k}"] = float(val)
 
                # 3. z_slope: linear slope across all real visits per latent dim
                #    This captures the TRAJECTORY — most important for AD progression
                real_z = z_visits[:slen, i, :]  # (slen, latent_dim)
                if slen >= 2:
                    visit_times = times[i, :slen].numpy()  # actual time values
                    dt = visit_times - visit_times[0]
                    if dt[-1] > 0:
                        # Weighted least-squares slope for each latent dim
                        for k in range(real_z.shape[1]):
                            z_vals = real_z[:, k]
                            # Simple linear slope: cov(t,z)/var(t)
                            slope = np.cov(dt, z_vals)[0, 1] / (np.var(dt) + 1e-8)
                            row[f"z_slope_{k}"] = float(slope)
                    else:
                        for k in range(real_z.shape[1]):
                            row[f"z_slope_{k}"] = 0.0
                else:
                    for k in range(real_z.shape[1]):
                        row[f"z_slope_{k}"] = 0.0
 
                rows.append(row)
 
    df = pd.DataFrame(rows).sort_values("PTID").reset_index(drop=True)
    df.to_csv(output_path, index=False)
 
    n_z_final = len([c for c in df.columns if c.startswith("z_final_")])
    n_z_slope = len([c for c in df.columns if c.startswith("z_slope_")])
    print(f"\n✓ Patient-level export saved to {output_path}")
    print(f"  Patients      : {len(df)}")
    print(f"  z_final feats : {n_z_final}  (LSTM summary — use these in Cox)")
    print(f"  z_slope feats : {n_z_slope}  (trajectory signal)")
    print(f"  Events        : {df['event'].sum()} ({100*df['event'].mean():.1f}%)")
    return df

# ============================================================================
# MAIN
# ============================================================================
def main():
    print("=" * 80)
    print("ENHANCED MCI-TO-AD PREDICTION")
    print("=" * 80)

    device = CONFIG['device']
    print(f"\nDevice: {device}")

    # ------------------------------------------------------------------
    # Load data
    # ------------------------------------------------------------------
    print("\nLoading data...")
    manifest = pd.read_csv("./AD_Multimodal/TFN_AD/AD_Patient_Manifest.csv")

    print("\nLoading valid patient list...")
    with open('VALID_PATIENTS.pkl', 'rb') as f:
        VALID_PATIENTS = pickle.load(f)

    # Normalise types so string IDs match regardless of how the pkl was saved
    VALID_PATIENTS = set(str(p) for p in VALID_PATIENTS)
    manifest["PTID"] = manifest["PTID"].astype(str)

    print(f"Valid patients in pkl : {len(VALID_PATIENTS)}")
    overlap = set(manifest["PTID"].unique()) & VALID_PATIENTS
    print(f"Overlap with manifest : {len(overlap)}")
    if len(overlap) == 0:
        raise RuntimeError(
            "No PTID overlap between manifest and VALID_PATIENTS.pkl — "
            "check for type mismatches (int vs str) or ID format differences."
        )

    # ------------------------------------------------------------------
    # Transforms & Dataset (raw, unscaled)
    # ------------------------------------------------------------------
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    print("\nCreating dataset...")
    dataset = SequenceDataset(manifest, VALID_PATIENTS, transform,
                              max_seq_len=CONFIG['max_seq_len'])

    print(f"\nTotal sequences: {len(dataset)}")
    if len(dataset) == 0:
        raise RuntimeError(
            "Dataset is empty after processing. Common causes:\n"
            "  1. Image paths don't exist (check ./AD_Multimodal/ADNI_Dataset/)\n"
            "  2. No patients have ≥2 visits with valid images\n"
            "  3. engineer_features raised errors (check ⚠️ lines above)\n"
            "  4. No patients have 'MCI' in their DX sequence"
        )

    # ------------------------------------------------------------------
    # Train / val split  — split BEFORE fitting scaler
    # ------------------------------------------------------------------
    n_train = int(0.8 * len(dataset))
    n_val = len(dataset) - n_train
    if n_train == 0:
        raise RuntimeError(
            f"Dataset too small to split (size={len(dataset)}). "
            "Need at least 2 patients."
        )

    train_dataset, val_dataset = torch.utils.data.random_split(
        dataset, [n_train, n_val],
        generator=torch.Generator().manual_seed(42)
    )
    print(f"Train : {n_train}  |  Val : {n_val}")

    # ------------------------------------------------------------------
    # Fit scaler on TRAIN indices only — zero leakage from val
    # ------------------------------------------------------------------
    train_indices = list(train_dataset.indices)
    if len(train_indices) == 0:
        raise RuntimeError("train_dataset.indices is empty — this should not happen.")

    train_tabs = np.vstack([
        dataset.sequences[i]['tabs'][:dataset.sequences[i]['seq_len']]
        for i in train_indices
    ])
    scaler = StandardScaler()
    scaler.fit(train_tabs)

    # Apply to ALL sequences (train + val) so export inverse_transform works
    dataset.apply_scaler(scaler)
    print("Scaler fitted on train split only — no val leakage ✓")

    # ------------------------------------------------------------------
    # DataLoaders
    # ------------------------------------------------------------------
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                              shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'],
                            shuffle=False, num_workers=0)

    # ------------------------------------------------------------------
    # Model
    # ------------------------------------------------------------------
    sample_tabs = dataset.sequences[0]['tabs']
    tab_dim = sample_tabs.shape[1]

    print(f"\nTab feature dimension : {tab_dim}")
    print("Initializing model...")
    model = SupervisedTemporalFusionAutoencoder(tab_dim, CONFIG).to(device)
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=CONFIG['lr'],
                                  weight_decay=CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5
    )

    # ------------------------------------------------------------------
    # Training loop
    # ------------------------------------------------------------------
    print("\n" + "=" * 80)
    print("TRAINING")
    print("=" * 80)

    best_c_index = 0.0
    patience_ctr = 0

    for epoch in range(CONFIG['epochs']):
        train_losses = train_epoch(model, train_loader, optimizer, CONFIG, device)
        val_c_index, _, _, _ = validate(model, val_loader, device)
        scheduler.step(val_c_index)

        print(f"\nEpoch {epoch + 1}/{CONFIG['epochs']}")
        print(f"  Loss : {train_losses['total']:.4f}  "
              f"(Cox: {train_losses['cox']:.4f}  MMSE: {train_losses['mmse']:.4f})")
        print(f"  Val C-index : {val_c_index:.4f}")

        if val_c_index > best_c_index:
            best_c_index = val_c_index
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'c_index': best_c_index,
            }, 'best_model.pth')
            print(f"  ✓ New best: {best_c_index:.4f}")
            patience_ctr = 0
        else:
            patience_ctr += 1

        if patience_ctr >= 15:
            print("\n⚠  Early stopping triggered.")
            break

    # ------------------------------------------------------------------
    # Export
    # ------------------------------------------------------------------
    print("\n" + "=" * 80)
    print("EXPORTING")
    print("=" * 80)

    checkpoint = torch.load('best_model.pth', weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"\nLoaded best model (epoch {checkpoint['epoch'] + 1}, "
          f"C-index {checkpoint['c_index']:.4f})")

    full_loader = DataLoader(dataset, batch_size=CONFIG['batch_size'],
                             shuffle=False, num_workers=0)

    latent_df = export_latent_features(
        model, full_loader, device,
        "latent_improved_autoencoder.csv"
    )

    patient_df = export_patient_level_latent(model, full_loader, device, "latent_patient_level.csv")

    print("\n" + "=" * 80)
    print("✓ COMPLETE")
    print("=" * 80)
    print(f"\nBest C-index : {best_c_index:.4f}")
    print("Ready for JMBayes2 in R!")


if __name__ == "__main__":
    main()

ENHANCED MCI-TO-AD PREDICTION

Device: cuda

Loading data...

Loading valid patient list...
Valid patients in pkl : 161
Overlap with manifest : 161

Creating dataset...
  Processed : 161 valid patients
  Skipped (not in valid set) : 221
  Skipped (errors)           : 0

Total sequences: 161
Train : 128  |  Val : 33
Scaler fitted on train split only — no val leakage ✓

Tab feature dimension : 16
Initializing model...
Parameters: 48,508,003

TRAINING

Epoch 1/100
  Loss : 1.9021  (Cox: 2.2377  MMSE: 0.0000)
  Val C-index : 0.7540
  ✓ New best: 0.7540

Epoch 2/100
  Loss : 1.9314  (Cox: 2.2723  MMSE: 0.0000)
  Val C-index : 0.7540

Epoch 3/100
  Loss : 1.7649  (Cox: 2.0763  MMSE: 0.0000)
  Val C-index : 0.7249

Epoch 4/100
  Loss : 1.4892  (Cox: 1.7520  MMSE: 0.0000)
  Val C-index : 0.7379

Epoch 5/100
  Loss : 1.3596  (Cox: 1.5995  MMSE: 0.0000)
  Val C-index : 0.7961
  ✓ New best: 0.7961

Epoch 6/100
  Loss : 1.4768  (Cox: 1.7374  MMSE: 0.0000)
  Val C-index : 0.7702

Epoch 7/100
  Loss